In [ ]:
from moabb.paradigms import P300
from moabb.datasets import *
from sklearn.metrics import get_scorer

paradigm = P300(resample=48)
dataset= BNCI2014_008()

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)
scorer = get_scorer(paradigm.scoring)


In [ ]:
from sklearn.model_selection import StratifiedKFold

n_blocks=16
cv = StratifiedKFold(n_splits=32, shuffle=True, random_state=42)
n_blocks_grid = list(range(1,n_blocks+1))
theta_grid = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]

In [ ]:
job_args = []
for subject in dataset.subject_list:
    X, y, meta = paradigm.get_data(dataset=dataset, subjects=[subject], cache_config=cache_config,)
    for fold, (train_idc, test_idc) in enumerate(cv.split(X, y)):
        for theta in theta_grid:
            job_args.append((dataset.code, subject, fold, X,y, train_idc, test_idc, theta))

In [ ]:
from hoda.hoda import BTTDA
from hoda.classification import ZScore
from sklearn.preprocessing import StandardScaler
from classification_erp import make_clf, get_hoda_params, get_bttda_params
import tensorly as tl
import pandas as pd
import warnings

def eval_fold(dataset, subject, fold, X, y, train_idc, test_idc, theta):
    X_st = ZScore().fit(X[train_idc], y[train_idc]).transform(X,y) 
    
    hoda_params = get_hoda_params()
    hoda_params['theta'] = theta
    bttda = BTTDA(
        ranks=[None]*max(n_blocks_grid),
        hoda_params=hoda_params,
        verbose=False,        
    )
    
    bttda.fit(X_st[train_idc], y[train_idc])
    print(bttda.n_blocks_)
    clf = make_clf()
    result = []
    for n_blocks in n_blocks_grid:
        if n_blocks > bttda.n_blocks_:
            break
        Xt = bttda.transform(X_st, n_blocks=n_blocks)
        X_rec = bttda.inv_transform(Xt, n_blocks=n_blocks)
        try:
            clf.fit(Xt[train_idc], y[train_idc])
        except ValueError as e:
            warnings.warn(str(e))
            break
        result.append(dict(
            subject = subject,
            dataset = dataset,
            fold = fold,
            theta=theta,
            n_blocks=n_blocks,
            train_score = scorer(clf, Xt[train_idc], y[train_idc]),
            test_score = scorer(clf, Xt[test_idc], y[test_idc]),
            train_mse = tl.metrics.regression.MSE(X_st[train_idc], X_rec[train_idc]),
            test_mse = tl.metrics.regression.MSE(X_st[test_idc], X_rec[test_idc]),          
        ))
    return pd.DataFrame(result)

        

In [ ]:
import joblib
from joblib import Parallel, delayed
from hpc import create_cluster, create_client, TIMEOUT

with create_cluster(cluster='wice') as cluster, create_client(cluster) as client:
    with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
        results = Parallel(n_jobs=len(job_args), verbose=True)(delayed(eval_fold)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

In [ ]:
results.to_csv('results/gridsearch_erp.csv')

In [ ]:
results

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

df = results.groupby(['n_blocks', 'theta'])['test_score'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='test_score', color='theta', color_discrete_sequence=colors)
fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = results.groupby(['n_blocks', 'theta'])['test_mse'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='test_mse', color='theta', color_discrete_sequence=colors)
fig.show()